# 카테고리별 N 전달통로 최소 M5 실험: Dunnhumby seed 42

동일 실행에서 두 arm만 새로 학습합니다. 기준 arm은 `q_V 가격위치 표현 + 현재 M4`, 후보 arm은 여기에 `카테고리별로 배분한 q_N 표현`만 추가합니다. 각 장바구니는 카테고리 수와 무관하게 총 1의 빈도 질량을 가지며, 모집단 카테고리 비중을 빼 인기 카테고리 지름길을 줄입니다. 이미 구매한 정확한 상품은 표현에 넣지 않고 기존 신규상품 평가 마스크로 제외합니다.

이 실행은 `historical_development_days_684_690`의 seed 42 방향성 확인입니다. 후보가 두 Top-10 경제지표에서 기준을 모두 이길 때만 카테고리-only 및 N 순열 대조군을 다음 단계에서 추가합니다. 이번 결과만으로 CLV 귀속·유의성·일반화를 주장하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import importlib, os, shutil, subprocess, sys

REVIEWED_SHA = '42b3825d165babba57877fab7e5dc9c537f0e81c'
REPO_URL = 'https://github.com/jung-un/clv-m2-lightgcn-runner.git'
os.chdir('/content')
repo = Path('/content/clv-m2-lightgcn-runner')
clone_errors = []
for clone_attempt in range(1, 4):
    os.chdir('/content')
    if repo.exists():
        shutil.rmtree(repo)
    result = subprocess.run(
        ['git', 'clone', REPO_URL, str(repo)],
        text=True, capture_output=True,
    )
    if result.returncode == 0:
        break
    clone_errors.append(result.stderr.strip())
    print(f'GitHub clone {clone_attempt}/3 실패:', result.stderr.strip())
else:
    raise RuntimeError('GitHub clone 3회 실패:\n' + '\n'.join(clone_errors))
subprocess.run(['git', '-C', str(repo), 'checkout', '-q', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(
    ['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True
).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
for module_name in tuple(sys.modules):
    if module_name.startswith(('lightgcn_', 'clv_')):
        del sys.modules[module_name]
importlib.invalidate_caches()
%cd /content/clv-m2-lightgcn-runner
print('실행 코드 고정 완료:', actual_sha)

In [ ]:
import json
import torch
from lightgcn_clv_m5_category_frequency_value_screen import (
    MODEL_IDS,
    configure_category_frequency_value_screen,
    preflight_summary,
    run_category_frequency_value_screen,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_category_frequency_value_screen(
    out_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m5_category_frequency_n_value_basis_'
        'two_arm_development_screen_v1'
    ),
)
summary = preflight_summary(cfg)
assert summary['split'] == 'historical_development_days_684_690'
assert summary['trained_models'] == list(MODEL_IDS)
assert summary['reused_models'] == []
assert summary['prior_result_file_required'] is False
assert summary['fixed']['new_item_task'] is True
assert summary['fixed']['train_pairs_excluded_from_evaluation'] is True
assert summary['fixed']['min_item_interactions'] == 1
assert summary['fixed']['graph'] == 'binary'
assert summary['fixed']['negative_sampling'] == 'uniform'
assert summary['fixed']['final_test_constructed'] is False
assert summary['fixed']['holdout_constructed'] is False
assert summary['fixed']['one_training_loop_and_optimizer_per_arm'] is True
assert summary['fixed']['external_reranking'] is False
assert summary['fixed']['m3_edge_weight'] is False
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
result_df = run_category_frequency_value_screen(cfg)

In [ ]:
from IPython.display import display

def show(frame):
    view = frame.copy()
    view.attrs = {}
    display(view)

print('1) q_V+M4 기준과 카테고리별 N+q_V+M4 절대지표')
show(result_df)
print('2) 카테고리별 N 증분 전체 지표 비교')
show(result_df.attrs['comparison'])
print('3) ID·V·카테고리별 N 점수 영향력')
show(result_df.attrs['score_diagnostics'])
print('4) 카테고리별 N 입력 진단')
print(json.dumps(result_df.attrs['category_frequency_diagnostics'], ensure_ascii=False, indent=2))
print('5) Top-10 변경 진단')
print(json.dumps(result_df.attrs['ranking_change'], ensure_ascii=False, indent=2))
print('6) 사전 판독')
print(json.dumps(result_df.attrs['decision'], ensure_ascii=False, indent=2))
print('7) 저장 파일')
print(json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))